<a href="https://colab.research.google.com/github/erickodiwuor014-max/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duka Electronics: Sales Exploration (January to June 2026)
Engagement 1 of `plp-da-a3-duka-sales-and-agent-cleanup`. Data: `duka_sales.csv` (1,000 orders, generated by `generate_data.py`, seed 2026).

## Executive Summary
- Duka Electronics earned KES 71,792,500 from 1,000 orders between January and June 2026, an average of KES 71,793 per order.
- Nairobi is the largest branch with KES 26,106,100 (36.4% of revenue) from 376 orders, and all five branches rank the same by revenue and by orders.
- Laptops generate 57.9% of revenue (KES 41,535,000) even though tablets sell more units (691 against 639).
- Monthly revenue peaked at KES 13,518,300 in April and fell to KES 10,903,400 in June, a 24.0% gap between the best and worst month.
- Weekends bring in 27.8% of revenue, and a weekend day earns KES 383,390 against KES 405,127 on a weekday, only about 5% less.

In [1]:
import os, sys, subprocess

REPO = "plp-da-a3-duka-sales-and-agent-cleanup"
if not os.path.exists("generate_data.py"):
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/erickodiwuor014-max/{REPO}.git"], check=True)
    os.chdir(REPO)
if not os.path.exists("duka_sales.csv"):
    subprocess.run([sys.executable, "generate_data.py"], check=True)

import numpy as np
import pandas as pd

print("Working folder:", os.getcwd())
print("Files:", sorted(os.listdir()))

Working folder: /content/plp-da-a3-duka-sales-and-agent-cleanup
Files: ['.git', '.gitignore', 'README.md', 'agent_transactions.csv', 'agents.csv', 'duka_sales.csv', 'duka_sales_exploration.ipynb', 'excel', 'generate_data.py', 'outputs', 'requirements.txt']


## 1. First-five-minutes inspection
Loading the data with `order_date` parsed as a real date, then checking shape, types, missing values, duplicates, date range and the key categories.

In [2]:
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("Shape (rows, columns):", df.shape)
df.head()

Shape (rows, columns): (1000, 7)


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


In [3]:
print(df.dtypes)
print()
df.info()

order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


In [4]:
print("Missing values per column:")
print(df.isna().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate order_id values:", df["order_id"].duplicated().sum())

Missing values per column:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Duplicate rows: 0
Duplicate order_id values: 0


In [5]:
print("Date range:", df["order_date"].min().date(), "to", df["order_date"].max().date())
print()
print(df[["quantity", "unit_price"]].describe())

Date range: 2026-01-01 to 2026-06-29

          quantity    unit_price
count  1000.000000   1000.000000
mean      3.012000  23918.300000
std       1.416991  23908.790546
min       1.000000   1200.000000
25%       2.000000   2500.000000
50%       3.000000  15000.000000
75%       4.000000  28000.000000
max       5.000000  65000.000000


In [6]:
for col in ["branch", "product", "channel"]:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

--- branch ---
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64

--- product ---
product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184
Name: count, dtype: int64

--- channel ---
channel
In-store       579
Online         320
Phone order    101
Name: count, dtype: int64



In [7]:
print("Missing values per column:")
print(df.isna().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate order_id values:", df["order_id"].duplicated().sum())
print()
print("Date range:", df["order_date"].min().date(), "to", df["order_date"].max().date())
print()
print(df["channel"].value_counts())
print()
print(df["product"].value_counts())

Missing values per column:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Duplicate rows: 0
Duplicate order_id values: 0

Date range: 2026-01-01 to 2026-06-29

channel
In-store       579
Online         320
Phone order    101
Name: count, dtype: int64

product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184
Name: count, dtype: int64


**Findings (first-five-minutes inspection).**

- **Shape:** 1,000 orders and 7 columns: `order_id`, `order_date`, `branch`, `product`, `quantity`, `channel`, `unit_price`.
- **Types:** `order_id`, `quantity` and `unit_price` are integers. `order_date` is read as text by default, so it was parsed with `parse_dates` and is now a proper datetime column. `branch`, `product` and `channel` are text categories.
- **Missing values:** none in any column.
- **Duplicates:** 0 duplicate rows and 0 duplicate `order_id` values, so every row is a distinct order.
- **Date range:** 2026-01-01 to 2026-06-29, which is 180 days. The data stops on 29 June, so June has 29 days of orders against 30 in the calendar, a small effect (about 3%) to bear in mind when comparing June with other months.
- **Branches (5):** Nairobi, Mombasa, Nakuru, Kisumu, Eldoret.
- **Channels (3):** In-store 579 orders (57.9%), Online 320 (32.0%), Phone order 101 (10.1%).
- **Products (5):** Laptop 220 orders, Tablet 215, Phone 194, Headphones 187, Charger 184. Orders are spread fairly evenly across products (18.4% to 22.0% each), so any large revenue difference between products must come from price, not from how often they are ordered.
- **Conclusion:** the data is complete and clean, so no cleaning was needed before analysis.

## 2. Derived columns
- `revenue` = `quantity` × `unit_price`
- `month` and `weekday` are taken from `order_date`
- `is_weekend` is True for Saturday and Sunday
- `price_band` is **Premium** when the product's `unit_price` is KES 28,000 or more, otherwise **Value** (the threshold is inclusive, so a KES 28,000 item is Premium)

In [8]:
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.month_name()
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

df[["order_id", "order_date", "branch", "product", "quantity", "unit_price",
    "revenue", "month", "weekday", "is_weekend", "price_band"]].head()

,order_id,order_date,branch,product,quantity,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,2500,7500,March,Saturday,True,Value


In [9]:
print("Total revenue (KES):", f"{df['revenue'].sum():,}")
print()
print(df["price_band"].value_counts())
print()
print(df.groupby("price_band")["unit_price"].agg(["min", "max"]))

Total revenue (KES): 71,792,500

price_band
Value      565
Premium    435
Name: count, dtype: int64

              min    max
price_band              
Premium     28000  65000
Value        1200  15000


## 3. Task 3: Branch ranking by revenue and by number of orders

In [10]:
branch = df.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    orders=("order_id", "count"),
    units=("quantity", "sum"),
).reset_index()

branch["avg_order_revenue"] = (branch["total_revenue"] / branch["orders"]).round(0)
branch["revenue_share_pct"] = (branch["total_revenue"] / branch["total_revenue"].sum() * 100).round(1)
branch["revenue_rank"] = branch["total_revenue"].rank(ascending=False, method="min").astype(int)
branch["orders_rank"] = branch["orders"].rank(ascending=False, method="min").astype(int)

branch = branch.sort_values("revenue_rank").reset_index(drop=True)
display(branch)

by_rev = branch.sort_values("total_revenue", ascending=False)["branch"].tolist()
by_orders = branch.sort_values("orders", ascending=False)["branch"].tolist()
print("Ranking by revenue:", by_rev)
print("Ranking by orders: ", by_orders)
print("Same ranking?", by_rev == by_orders)

,branch,total_revenue,orders,units,avg_order_revenue,revenue_share_pct,revenue_rank,orders_rank
0,Nairobi,26106100,376,1107,69431.0,36.4,1,1
1,Mombasa,14003700,186,567,75289.0,19.5,2,2
2,Nakuru,11970400,162,495,73891.0,16.7,3,3
3,Kisumu,10868100,159,473,68353.0,15.1,4,4
4,Eldoret,8844200,117,370,75591.0,12.3,5,5


Ranking by revenue: ['Nairobi', 'Mombasa', 'Nakuru', 'Kisumu', 'Eldoret']
Ranking by orders:  ['Nairobi', 'Mombasa', 'Nakuru', 'Kisumu', 'Eldoret']
Same ranking? True


**Answer (Task 3).** Yes, the two rankings are identical. Both revenue and order count rank the branches Nairobi, Mombasa, Nakuru, Kisumu, Eldoret.

| Branch | Orders | Share of orders | Revenue (KES) | Share of revenue | Avg order (KES) |
|---|---|---|---|---|---|
| Nairobi | 376 | 37.6% | 26,106,100 | 36.4% | 69,431 |
| Mombasa | 186 | 18.6% | 14,003,700 | 19.5% | 75,289 |
| Nakuru | 162 | 16.2% | 11,970,400 | 16.7% | 73,891 |
| Kisumu | 159 | 15.9% | 10,868,100 | 15.1% | 68,353 |
| Eldoret | 117 | 11.7% | 8,844,200 | 12.3% | 75,591 |

**Why they match.** Average order value is similar everywhere, from KES 68,353 to KES 75,591 (a spread of about 11%), so a branch's revenue mostly follows how many orders it takes. Nairobi has 376 orders, twice Mombasa's 186, and that volume gives it first place. Eldoret has the highest average order (KES 75,591) but the fewest orders (117), so larger baskets cannot lift it above the others.

**What the shares add.** Nairobi's revenue share (36.4%) is slightly below its order share (37.6%) because its average order is below Mombasa, Nakuru and Eldoret. Its lead comes from volume, not bigger sales. Mombasa shows the opposite, with 19.5% of revenue from 18.6% of orders. Eldoret and Kisumu are the branches with the most to gain from more orders.

## 4. Task 4: Top three products by revenue and by units sold

In [11]:
product = df.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    units_sold=("quantity", "sum"),
    orders=("order_id", "count"),
    unit_price=("unit_price", "mean"),
).reset_index()

top_rev = product.nlargest(3, "total_revenue")
top_units = product.nlargest(3, "units_sold")

print("Top 3 by revenue:")
display(top_rev)
print("Top 3 by units sold:")
display(top_units)

print("Only in revenue top 3:", set(top_rev["product"]) - set(top_units["product"]))
print("Only in units top 3:  ", set(top_units["product"]) - set(top_rev["product"]))

Top 3 by revenue:


,product,total_revenue,units_sold,orders,unit_price
2,Laptop,41535000,639,220,65000.0
4,Tablet,19348000,691,215,28000.0
3,Phone,8880000,592,194,15000.0


Top 3 by units sold:


,product,total_revenue,units_sold,orders,unit_price
4,Tablet,19348000,691,215,28000.0
2,Laptop,41535000,639,220,65000.0
3,Phone,8880000,592,194,15000.0


Only in revenue top 3: set()
Only in units top 3:   set()


**Answer (Task 4).** The top three products are the same by revenue and by units, in a different order.

| Rank | By revenue | Revenue (KES) | Share | By units | Units |
|---|---|---|---|---|---|
| 1 | Laptop | 41,535,000 | 57.9% | Tablet | 691 |
| 2 | Tablet | 19,348,000 | 27.0% | Laptop | 639 |
| 3 | Phone | 8,880,000 | 12.4% | Phone | 592 |

**Why the order differs.** Tablets sold 52 more units than laptops (691 against 639), but a laptop costs KES 65,000 and a tablet KES 28,000. Each laptop earns about 2.3 times as much, so 639 laptops bring in KES 41,535,000 against KES 19,348,000 for tablets. Phones are third in both lists because their price (KES 15,000) is the lowest of the three.

**Business meaning.** These three products make up 97.2% of revenue (KES 69,763,000). Headphones and chargers account for about 1,090 of the 3,012 units sold (36%) but only 2.8% of revenue, because they cost KES 2,500 and KES 1,200. Laptop sales are the biggest revenue risk, with more than half of all revenue.

## 5. Task 5: Weekend and weekday orders

In [12]:
wk = df.groupby("is_weekend").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_order_revenue=("revenue", "mean"),
).round(0)
wk.index = wk.index.map({False: "Weekday (Mon-Fri)", True: "Weekend (Sat-Sun)"})

all_days = pd.date_range(df["order_date"].min(), df["order_date"].max())
days_count = pd.Series(all_days.dayofweek >= 5).map({False: "Weekday (Mon-Fri)", True: "Weekend (Sat-Sun)"}).value_counts()
wk["calendar_days"] = days_count
wk["orders_per_day"] = (wk["orders"] / wk["calendar_days"]).round(1)
wk["revenue_per_day"] = (wk["total_revenue"] / wk["calendar_days"]).round(0)
display(wk)

,orders,total_revenue,avg_order_revenue,calendar_days,orders_per_day,revenue_per_day
is_weekend,,,,,,
Weekday (Mon-Fri),723,51856200,71724.0,128,5.6,405127.0
Weekend (Sat-Sun),277,19936300,71972.0,52,5.3,383390.0


**Answer (Task 5).** Weekday orders are far more numerous in total, but the two groups perform almost the same once you allow for the number of days.

| | Orders | Total revenue (KES) | Avg order (KES) | Calendar days | Orders per day | Revenue per day (KES) |
|---|---|---|---|---|---|---|
| Weekday (Mon-Fri) | 723 | 51,856,200 | 71,724 | 128 | 5.6 | 405,127 |
| Weekend (Sat-Sun) | 277 | 19,936,300 | 71,972 | 52 | 5.3 | 383,390 |

Weekdays account for 72.3% of orders and 72.2% of revenue. This is mostly because they make up 128 of the 180 days (71.1%), not because customers buy more on them.

**Fair comparison.** Per day, weekends are only slightly quieter: 5.3 orders against 5.6 (about 5% fewer) and KES 383,390 against KES 405,127 in revenue (about 5% lower). The average weekend order is KES 71,972, which is KES 248 (0.3%) higher than the weekday order of KES 71,724. Customers therefore spend the same amount per order on any day, and the small weekend gap comes from slightly fewer orders. Duka should staff and stock weekends almost as it does weekdays, because they carry 27.8% of revenue.

## 6. Task 6: Best and worst month for revenue

In [13]:
monthly = df.groupby(df["order_date"].dt.to_period("M")).agg(
    revenue=("revenue", "sum"), orders=("order_id", "count")
)
display(monthly)

best_m, worst_m = monthly["revenue"].idxmax(), monthly["revenue"].idxmin()
best_v, worst_v = monthly["revenue"].max(), monthly["revenue"].min()
print("Best month: ", best_m, f"KES {best_v:,}")
print("Worst month:", worst_m, f"KES {worst_v:,}")
print(f"Best month is {(best_v - worst_v) / worst_v * 100:.1f}% higher than the worst month")
print(f"Worst month is {(best_v - worst_v) / best_v * 100:.1f}% lower than the best month")

,revenue,orders
order_date,,
2026-01,11786600,173
2026-02,11772000,159
2026-03,12396800,186
2026-04,13518300,167
2026-05,11415400,153
2026-06,10903400,162


Best month:  2026-04 KES 13,518,300
Worst month: 2026-06 KES 10,903,400
Best month is 24.0% higher than the worst month
Worst month is 19.3% lower than the best month


**Answer (Task 6).** The best month for revenue was **April 2026** at KES 13,518,300, and the worst was **June 2026** at KES 10,903,400. April was **24.0% higher** than June, or equivalently June was **19.3% lower** than April. The gap is KES 2,614,900.

**Context.** Average monthly revenue was about KES 11,965,000, so April was 13.0% above average and June 8.9% below. The gap is not explained by order volume. April had 167 orders against 162 in June, but its average order was about KES 80,900 against KES 67,300, which points to more high-priced items in the mix. March had the most orders (186) but ranked second for revenue. Revenue then fell in May and June, so the cause of the decline should be checked before the half-year review.

## 7. Task 7: Premium online orders from Nairobi and Mombasa

In [14]:
print("Channel values:", df["channel"].unique())

Channel values: ['In-store' 'Online' 'Phone order']


In [15]:
mask = (
    (df["price_band"] == "Premium")
    & (df["channel"] == "Online")
    & (df["branch"].isin(["Nairobi", "Mombasa"]))
)
premium_coast_capital = df[mask]

print("Number of orders:", len(premium_coast_capital))
print("Total revenue (KES):", f"{premium_coast_capital['revenue'].sum():,}")
display(premium_coast_capital.groupby("branch")["revenue"].agg(["count", "sum"]))

os.makedirs("outputs", exist_ok=True)
premium_coast_capital.to_csv("outputs/premium_online_coast_capital.csv", index=False)
print(pd.read_csv("outputs/premium_online_coast_capital.csv").shape)

Number of orders: 83
Total revenue (KES): 10,218,000


,count,sum
branch,,
Mombasa,33,3696000
Nairobi,50,6522000


(83, 12)


In [16]:
from google.colab import files
files.download("outputs/premium_online_coast_capital.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
from google.colab import files

cols = ["order_id", "order_date", "branch", "product", "quantity", "channel", "unit_price", "revenue"]
export = df[cols].copy()
export["order_date"] = export["order_date"].dt.date
export.to_excel("duka_data_for_excel.xlsx", sheet_name="Data", index=False)

print("Branches:", sorted(df["branch"].unique()))
print(df.groupby("branch")["revenue"].sum().sort_values(ascending=False))
files.download("duka_data_for_excel.xlsx")

Branches: ['Eldoret', 'Kisumu', 'Mombasa', 'Nairobi', 'Nakuru']
branch
Nairobi    26106100
Mombasa    14003700
Nakuru     11970400
Kisumu     10868100
Eldoret     8844200
Name: revenue, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Answer (Task 7).** There are **83** Premium online orders from Nairobi and Mombasa, with a combined revenue of **KES 10,218,000**. That is 8.3% of all 1,000 orders but 14.2% of total revenue (KES 71,792,500), because Premium items sell at KES 28,000 or more.

| Branch | Orders | Revenue (KES) | Share of the group | Average per order (KES) |
|---|---|---|---|---|
| Nairobi | 50 | 6,522,000 | 63.8% | 130,440 |
| Mombasa | 33 | 3,696,000 | 36.2% | 112,000 |

Nairobi supplies about three in five of these orders and earns more per order. The average of KES 123,108 per order is roughly 1.7 times the overall average order of KES 71,793. The filtered orders were saved to `outputs/premium_online_coast_capital.csv` with `index=False`.

## 8. Spreadsheet check: Excel totals vs pandas
The branch revenue table in `excel/duka_branch_summary.xlsx` was built with `SUMIFS`, the region column with `XLOOKUP`, and a PivotTable shows revenue by branch and channel. Below, the spreadsheet values are read back and compared with the pandas totals.

In [18]:
xl = pd.read_excel("excel/duka_branch_summary.xlsx", sheet_name="Branch Summary")
xl = xl[xl["Branch"] != "Total"].dropna(subset=["Branch"])

pd_totals = df.groupby("branch")["revenue"].sum()
compare = xl.set_index("Branch")[["Revenue (SUMIFS)"]].join(pd_totals.rename("pandas_revenue"))
compare["match"] = compare["Revenue (SUMIFS)"] == compare["pandas_revenue"]
display(compare)
print("All branch totals match:", compare["match"].all())
print("Grand total pandas:", f"{df['revenue'].sum():,}")

,Revenue (SUMIFS),pandas_revenue,match
Branch,,,
Nairobi,26106100,26106100,True
Mombasa,14003700,14003700,True
Nakuru,11970400,11970400,True
Kisumu,10868100,10868100,True
Eldoret,8844200,8844200,True


All branch totals match: True
Grand total pandas: 71,792,500


In [19]:
%cd /content/plp-da-a3-duka-sales-and-agent-cleanup
!rm -f outputs/premium_online_coast_capital.csv
!git pull
!ls excel outputs

/content
Already up to date.
excel:
duka_branch_summary.xlsx

outputs:


In [20]:
xl = pd.read_excel("excel/duka_branch_summary.xlsx", sheet_name="Branch Summary")
xl = xl[xl["Branch"] != "Total"].dropna(subset=["Branch"])

pd_totals = df.groupby("branch")["revenue"].sum()
compare = xl.set_index("Branch")[["Revenue (SUMIFS)"]].join(pd_totals.rename("pandas_revenue"))
compare["match"] = compare["Revenue (SUMIFS)"] == compare["pandas_revenue"]
display(compare)
print("All branch totals match:", compare["match"].all())
print("Grand total pandas:", f"{df['revenue'].sum():,}")

,Revenue (SUMIFS),pandas_revenue,match
Branch,,,
Nairobi,26106100,26106100,True
Mombasa,14003700,14003700,True
Nakuru,11970400,11970400,True
Kisumu,10868100,10868100,True
Eldoret,8844200,8844200,True


All branch totals match: True
Grand total pandas: 71,792,500
